# DeepTaxa – Tutorial 1: Prediction
**DeepTaxa** is a hybrid CNN-BERT model for hierarchical 16S rRNA taxonomic classification
across seven ranks: Domain → Phylum → Class → Order → Family → Genus → Species.

This notebook reproduces the [official prediction tutorial](https://systems-genomics-lab.github.io/deeptaxa/prediction.html).

In [ ]:
# Install DeepTaxa and dependencies
!pip install -q deeptaxa huggingface_hub

In [ ]:
# Authenticate with HuggingFace (model weights are gated)
from huggingface_hub import login
login()  # enter your HF token when prompted

In [ ]:
import os, subprocess, json
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)

In [ ]:
# Download the pretrained model checkpoint
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='deeptaxa-full-length-v1.pt',
    local_dir='models'
)
print('Model saved to:', model_path)

In [ ]:
# Download example 16S sequences (FASTA)
!wget -q -O data/sample_sequences.fna \
  https://raw.githubusercontent.com/systems-genomics-lab/deeptaxa/main/data/sample_sequences.fna
!head -4 data/sample_sequences.fna

In [ ]:
# Inspect the FASTA file
from Bio import SeqIO
records = list(SeqIO.parse('data/sample_sequences.fna', 'fasta'))
print(f'Number of sequences: {len(records)}')
for r in records[:3]:
    print(f'  {r.id}: {len(r.seq)} bp')

In [ ]:
# Run predictions (CLI)
!deeptaxa predict \
    --model-path models/deeptaxa-full-length-v1.pt \
    --input data/sample_sequences.fna \
    --output-dir outputs/predictions \
    --batch-size 8

In [ ]:
# Load and display prediction results
import glob, pandas as pd
pred_files = glob.glob('outputs/predictions/*.json')
with open(pred_files[0]) as f:
    preds = json.load(f)
df = pd.DataFrame(preds)
df.head()

In [ ]:
# Prediction confidence per rank
import matplotlib.pyplot as plt
ranks = ['domain','phylum','class','order','family','genus','species']
if 'confidence' in df.columns:
    conf_cols = [c for c in df.columns if 'confidence' in c.lower()]
    means = [df[c].mean() for c in conf_cols]
    plt.figure(figsize=(8,4))
    plt.bar(ranks[:len(means)], means, color='steelblue')
    plt.ylabel('Mean confidence')
    plt.title('DeepTaxa prediction confidence by rank')
    plt.tight_layout(); plt.show()

In [ ]:
# Predict on the full Greengenes2 test set (optional – requires download)
# !wget -q -O data/gg2_test.fna <URL_TO_TEST_SET>
# !deeptaxa predict --model-path models/deeptaxa-full-length-v1.pt \
#     --input data/gg2_test.fna --output-dir outputs/predictions --batch-size 32

In [ ]:
# Predict on your own sequences
custom_fasta = ">my_seq\nAGAGTTTGATCCTGGCTCAGGATGAACGCTGGCGGCGTGCCTAATACATGCAAGTC\n"
with open('data/custom.fna', 'w') as f: f.write(custom_fasta)
!deeptaxa predict \
    --model-path models/deeptaxa-full-length-v1.pt \
    --input data/custom.fna \
    --output-dir outputs/predictions

In [ ]:
with open('outputs/predictions/custom_deeptaxa_predictions.json') as f:
    custom_preds = json.load(f)
import pprint; pprint.pprint(custom_preds)